In [11]:
# scripts/prepare_dataset.py
import os
import random
import json
import warnings
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
import soundfile as sf

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")
AUDIO_DIR = ROOT / "data" / "original_audio"
CSV_DIR = ROOT / "data"

OUTPUT_ROOT = ROOT / "processed_data"
OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)

TEST_FILES = set(range(1, 9))           # 1~8 固定测试集
EXCLUDE_FILES = {12}                     # 无pulse信号，排除

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def get_audio_files() -> List[Path]:
    return sorted(AUDIO_DIR.glob("*.wav"), key=lambda x: int(x.stem[-2:]))


def parse_file_num(filename: str) -> int:
    stem = Path(filename).stem
    try:
        return int(stem[-2:])
    except:
        return -1


def load_train_csvs():
    click = pd.read_csv(CSV_DIR / "ClickTrains.csv")
    burst = pd.read_csv(CSV_DIR / "BurstPulseTrains.csv")
    buzz  = pd.read_csv(CSV_DIR / "BuzzTrains.csv")

    all_dfs = [click, burst, buzz]
    for df in all_dfs:
        df.columns = df.columns.str.strip()

    all_trains = pd.concat(all_dfs, ignore_index=True)

    # 使用你实际打印出来的列名（完全匹配）
    required_cols = ['Ori_file_num(No.)', 'Train_start(s)', 'Train_end(s)']
    
    # 检查是否所有必要列都存在
    missing = [col for col in required_cols if col not in all_trains.columns]
    if missing:
        raise ValueError(f"缺少列: {missing}\n实际列名: {list(all_trains.columns)}")

    # 只保留需要的三列，并重命名
    all_trains = all_trains[required_cols].copy()
    all_trains = all_trains.rename(columns={
        'Ori_file_num(No.)': 'file_num',
        'Train_start(s)': 'start',
        'Train_end(s)': 'end'
    })

    # 强制转换为 float（避免后续比较出错）
    all_trains['file_num'] = all_trains['file_num'].astype(int)
    all_trains['start'] = all_trains['start'].astype(float)
    all_trains['end'] = all_trains['end'].astype(float)

    print("重命名后的列名:", list(all_trains.columns))
    print("前3行示例:\n", all_trains.head(3))

    return all_trains


# ----------------------- 1. 数据划分 -----------------------
def split_dataset():
    all_files = get_audio_files()
    file_map = {parse_file_num(f.name): f for f in all_files if parse_file_num(f.name) > 0}

    # 测试集 1~8
    test_files = [file_map[i] for i in range(1, 9) if i in file_map]

    # 训练/验证候选：9~35 排除12
    trainval_nums = [i for i in range(9, 36) if i not in EXCLUDE_FILES and i in file_map]
    trainval_files = [file_map[i] for i in trainval_nums]

    print(f"测试集文件数: {len(test_files)} (Ori 1-8)")
    print(f"训练/验证文件数: {len(trainval_files)} (排除12后)")

    # 5折
    random.shuffle(trainval_files)
    folds = np.array_split(trainval_files, 5)

    split_info = {
        "test": [str(p) for p in test_files],
        "folds": []
    }

    for i, fold_files in enumerate(folds, 1):
        val_file = fold_files[-1]
        train_files = fold_files[:-1].tolist()
        split_info["folds"].append({
            "fold": i,
            "train": [str(p) for p in train_files],
            "val": str(val_file)
        })

    with open(OUTPUT_ROOT / "data_split.json", "w", encoding="utf-8") as f:
        json.dump(split_info, f, ensure_ascii=False, indent=2)

    print("划分信息已保存: processed_data/data_split.json")
    return split_info


# ----------------------- 2. 生成 bag 标签 -----------------------
def generate_bag_labels():
    all_trains = load_train_csvs()
    
    print("all_trains columns after loading:", list(all_trains.columns))   
    print("Sample rows:\n", all_trains.head(3))
    trains_by_file = {k: g for k, g in all_trains.groupby('file_num')}

    all_files = get_audio_files()
    bag_records = []
    file_summary = []

    output_dir = OUTPUT_ROOT / "bag_labels"
    output_dir.mkdir(exist_ok=True, parents=True)

    for audio_path in all_files:
        file_num = parse_file_num(audio_path.name)
        if file_num <= 0:
            continue
        if file_num in EXCLUDE_FILES:
            continue        
        try:
            info = sf.info(str(audio_path))
            duration = info.duration
        except Exception as e:
            print(f"读取失败 {audio_path.name}: {e}")
            continue

        if duration < 60:
            print(f"跳过短文件 {audio_path.name} ({duration:.1f}s)")
            continue

        n_bags = int(duration // 60)
        file_trains = trains_by_file.get(file_num, pd.DataFrame())

        pos_count = 0
        total_train_sec = 0.0

        for i in range(n_bags):
            t_start = i * 60.0
            t_end = (i + 1) * 60.0

            overlap = file_trains[
                (file_trains['start'] < t_end) &
                (file_trains['end'] > t_start)
            ]

            label = 1 if len(overlap) > 0 else 0
            train_sec = 0.0

            if label == 1:
                for _, row in overlap.iterrows():
                    o_start = max(row['start'], t_start)
                    o_end = min(row['end'], t_end)
                    train_sec += o_end - o_start
                pos_count += 1
                total_train_sec += train_sec

            bag_records.append({
                "file_num": file_num,
                "audio_file": audio_path.name,
                "bag_idx": i,
                "bag_start_sec": t_start,
                "bag_end_sec": t_end,
                "label": label,
                "train_duration_sec": round(train_sec, 4),
                "audio_duration_sec": round(duration, 2)
            })

        file_summary.append({
            "file_num": file_num,
            "audio_file": audio_path.name,
            "duration_min": round(duration / 60, 1),
            "n_bags": n_bags,
            "positive_bags": pos_count,
            "total_train_sec": round(total_train_sec, 2)
        })

        # 每个文件单独保存一份（可选，方便调试）
        # pd.DataFrame([r for r in bag_records if r["file_num"] == file_num]).to_csv(
        #     output_dir / f"file_{file_num:02d}_bags.csv", index=False, encoding="utf-8-sig"
        # )

    # 所有bag汇总
    df_bags = pd.DataFrame(bag_records)
    df_bags.to_csv(OUTPUT_ROOT / "all_bags.csv", index=False, encoding="utf-8-sig")

    # 文件级别统计
    df_summary = pd.DataFrame(file_summary)
    df_summary.to_csv(OUTPUT_ROOT / "file_summary.csv", index=False, encoding="utf-8-sig")

    print(f"\n完成。共处理 {len(file_summary)} 个文件")
    print(f"总bag数量: {len(df_bags)}")
    print(f"主要输出文件：")
    print(f"  - {OUTPUT_ROOT}/all_bags.csv")
    print(f"  - {OUTPUT_ROOT}/file_summary.csv")


if __name__ == "__main__":
    print("=== 数据准备开始 ===")
    split_dataset()
    print("\n=== bag标签生成开始 ===")
    generate_bag_labels()
    print("=== 全部完成 ===")

=== 数据准备开始 ===
测试集文件数: 8 (Ori 1-8)
训练/验证文件数: 26 (排除12后)
划分信息已保存: processed_data/data_split.json

=== bag标签生成开始 ===
重命名后的列名: ['file_num', 'start', 'end']
前3行示例:
    file_num     start       end
0         1  278.2543  278.7155
1         1  296.8332  297.3997
2         1  360.1263  360.5878
all_trains columns after loading: ['file_num', 'start', 'end']
Sample rows:
    file_num     start       end
0         1  278.2543  278.7155
1         1  296.8332  297.3997
2         1  360.1263  360.5878

完成。共处理 34 个文件
总bag数量: 957
主要输出文件：
  - D:\Project_Github\audio_click_mil\processed_data/all_bags.csv
  - D:\Project_Github\audio_click_mil\processed_data/file_summary.csv
=== 全部完成 ===
